# API Routes - Simulation
Flask routes for simulation control.

In [ ]:
# Simulation API routes

def register_simulation_routes(app, simulation_state, simulation_lock, 
                                start_simulation_fn, stop_simulation_fn, 
                                reset_simulation_fn, set_speed_fn, get_tile_allocation_fn):
    """Register simulation-related API routes"""
    from flask import jsonify, request
    
    @app.route('/api/simulation/state', methods=['GET'])
    def get_simulation_state():
        """Get current simulation state"""
        with simulation_lock:
            return jsonify({
                "running": simulation_state['running'],
                "tick": simulation_state['tick'],
                "speed": simulation_state['speed'],
                "robots": simulation_state['robots'],
                "observation_queue": simulation_state['observation_queue'],
                "tile_allocation": get_tile_allocation_fn()
            }), 200

    @app.route('/api/simulation/start', methods=['POST'])
    def start_simulation():
        """Start the simulation"""
        success = start_simulation_fn()
        return jsonify({"success": success}), 200

    @app.route('/api/simulation/stop', methods=['POST'])
    def stop_simulation():
        """Stop the simulation"""
        success = stop_simulation_fn()
        return jsonify({"success": success}), 200

    @app.route('/api/simulation/reset', methods=['POST'])
    def reset_simulation():
        """Reset the simulation"""
        success = reset_simulation_fn()
        return jsonify({"success": success}), 200

    @app.route('/api/simulation/speed', methods=['PUT'])
    def set_simulation_speed():
        """Set simulation speed"""
        try:
            data = request.get_json()
            speed = data.get('speed', 1.0)
            
            if speed <= 0:
                return jsonify({"error": "Speed must be positive"}), 400
            
            set_speed_fn(speed)
            return jsonify({"success": True, "speed": speed}), 200
        except Exception as e:
            return jsonify({"error": str(e)}), 500